# 04 – Model Evaluation
Evaluates **persisted** predictions – no models are loaded.

In [1]:
# -------------------------------------------------------------
# Common pre‑amble – read config & export core variables
# -------------------------------------------------------------

from pathlib import Path
import os, sys

# infer repo root from the location of this file
repo_root = os.path.abspath('..')
sys.path.insert(0, str(repo_root))  # allow `import src.*`
from config.notebook_setup import *

Repository Root: /home/marcmaceira/projects/reuters-rag-classifier_clean
Configuration: {'general': {'run_name': 'experiment_with_13_classes', 'seed': 42, 'n_classes': 13}, 'dataset': {'split_type': 'test', 'test_split': 0.2, 'cutoff_year': 1996}, 'paths': {'data_exploration_dir': 'experiment_with_13_classes/data_exploration', 'artifacts_dir': 'experiment_with_13_classes/artifacts', 'embeddings_dir': 'experiment_with_13_classes/embeddings', 'models_dir': 'experiment_with_13_classes/models', 'results_dir': 'experiment_with_13_classes/results'}, 'model': {'embedding_backend': 'sbert', 'sbert_model_name': 'sentence-transformers/all-MiniLM-L12-v2', 'openai_model_name': 'text-embedding-3-small', 'classifier': 'linear_svm', 'rag_top_k': 5, 'use_llm_refine': False}, 'training': {'batch_size': 32, 'max_epochs': 10, 'learning_rate': '1e-3'}, 'evaluation': {'metrics': ['accuracy', 'macro_f1', 'auc_ovr']}}

=== Configuration Variables ===

[DATASET]
  DATASET_CUTOFF_YEAR: 1996
  DATASET_SPLIT_TYP

In [2]:

from pathlib import Path
from src.datasets.dataset import get_dataset
from src.evaluation import run_evaluations

# -------------------------------------------------------------
# 1. Load test labels
# -------------------------------------------------------------

# Load dataset
# X_train, y_train, _, _, _ = get_dataset(split_type="standard", n_classes=N_CLASSES)
X_train, y_train, X_test, y_test, classes = get_dataset(
    split_type=DATASET_SPLIT_TYPE,
    n_classes=N_CLASSES,
    cutoff_year=GENERAL_CUTOFF_YEAR
)

print(f"Loaded {len(X_train)} training documents with {N_CLASSES} classes")

# -------------------------------------------------------------
# 2. Discover model artefacts
# -------------------------------------------------------------
model_names = [p.name for p in Path(MODELS_DIR).iterdir() if (p / "test_predictions.csv").exists()]
print("Found models:", model_names)




INFO | Loading Reuters dataset with configuration:
INFO |   - Split type: test
INFO |   - Number of classes: 13
INFO |   - Samples per class: 20
INFO |   - Random seed: None
INFO | Loading small test dataset with 20 samples per class across 13 classes
INFO | Selected classes: earn, acq, crude, interest, money-fx, trade, grain, corn, dlr, money-supply, ship, coffee, sugar
INFO |   - Class 'earn': 14 train, 6 test
INFO |   - Class 'acq': 14 train, 6 test
INFO |   - Class 'crude': 14 train, 6 test
INFO |   - Class 'interest': 14 train, 6 test
INFO |   - Class 'money-fx': 14 train, 6 test
INFO |   - Class 'trade': 14 train, 6 test
INFO |   - Class 'grain': 14 train, 6 test
INFO |   - Class 'corn': 14 train, 6 test
INFO |   - Class 'dlr': 14 train, 6 test
INFO |   - Class 'money-supply': 14 train, 6 test
INFO |   - Class 'ship': 14 train, 6 test
INFO |   - Class 'coffee': 14 train, 6 test
INFO |   - Class 'sugar': 14 train, 6 test


Loaded 182 training documents with 13 classes
Found models: ['RAG-kMajority', 'Naive Bayes', 'RAG-LLM (OpenAI-embeddings)', 'MiniLM + LogReg', 'RAG-CentroidNN', 'RAG-LLM (local-embeddings)', 'TF-IDF bigrams + SVM', 'Linear SVM']


In [3]:
# -------------------------------------------------------------
# 3. Run evaluation
# -------------------------------------------------------------
results = run_evaluations(
    model_names,
    y_true=y_test,
    y_train_true=y_train,  # Pass training labels
    artefacts_root=MODELS_DIR,
    output_dir=RESULTS_DIR,
    verbose=True,
)

# Display detailed results
import pandas as pd
print("\n=== Summary of Test Metrics ===")
test_metrics = pd.DataFrame({
    model: {k: v for k, v in metrics.items() if k.startswith('test_')}
    for model, metrics in results.items()
}).T
display(test_metrics)

# Show train metrics if available (requires having train predictions saved)
train_cols = [col for col in next(iter(results.values())).keys() if col.startswith('train_')]
if train_cols:
    print("\n=== Summary of Train Metrics ===")
    train_metrics = pd.DataFrame({
        model: {k: v for k, v in metrics.items() if k.startswith('train_')}
        for model, metrics in results.items()
    }).T
    display(train_metrics)
    
    # Show potential overfitting metrics
    diff_cols = [col for col in next(iter(results.values())).keys() if col.endswith('_diff')]
    if diff_cols:
        print("\n=== Train/Test Differences (Overfitting Analysis) ===")
        diff_metrics = pd.DataFrame({
            model: {k: v for k, v in metrics.items() if k.endswith('_diff')}
            for model, metrics in results.items()
        }).T
        display(diff_metrics)
        
        # Interpretation guideline
        print("\nInterpretation guide:")
        print("- Positive values indicate potential overfitting (model performs better on training data)")
        print("- Values close to zero indicate good generalization")
        print("- Negative values might indicate underfitting or data leakage issues")

[run_evaluations] Processing RAG-kMajority...
[run_evaluations] Processing RAG-kMajority training metrics...
[run_evaluations] Processing Naive Bayes...
[run_evaluations] Processing Naive Bayes training metrics...
[run_evaluations] Processing RAG-LLM (OpenAI-embeddings)...
[run_evaluations] Processing RAG-LLM (OpenAI-embeddings) training metrics...


/home/marcmaceira/projects/reuters-rag-classifier_clean/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/marcmaceira/projects/reuters-rag-classifier_clean/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/marcmaceira/projects/reuters-rag-classifier_clean/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` paramete

[run_evaluations] Processing MiniLM + LogReg...
[run_evaluations] Processing MiniLM + LogReg training metrics...
[run_evaluations] Processing RAG-CentroidNN...
[run_evaluations] Processing RAG-CentroidNN training metrics...
[run_evaluations] Processing RAG-LLM (local-embeddings)...


/home/marcmaceira/projects/reuters-rag-classifier_clean/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/marcmaceira/projects/reuters-rag-classifier_clean/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/marcmaceira/projects/reuters-rag-classifier_clean/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` paramete

[run_evaluations] Processing RAG-LLM (local-embeddings) training metrics...
[run_evaluations] Processing TF-IDF bigrams + SVM...
[run_evaluations] Processing TF-IDF bigrams + SVM training metrics...
[run_evaluations] Processing Linear SVM...
[run_evaluations] Processing Linear SVM training metrics...

=== Summary of Test Metrics ===


...

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>